In [1]:
# Cell 1 - install dependencies
# Direct imports: torch, transformers, peft, trl, datasets, huggingface_hub
# Installed but used internally: bitsandbytes (quantization), accelerate (device management)
!pip install transformers bitsandbytes accelerate peft trl datasets huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 111.5 MB/s eta 0:00:00
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65

In [2]:
# Cell 2 - Authenticate with Hugging Face
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret('HF_TOKEN')

if hf_token:
  login(token=hf_token)


In [3]:
# Cell 3 - Confirm GPU
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

CUDA available: True
Device: Tesla T4
VRAM: 15.6 GB


In [4]:
import os

os.makedirs("/kaggle/working/checkpoints", exist_ok=True)
print("Checkpoint directory ready.")

Checkpoint directory ready.


In [5]:
# Cell - 4 Load Cleaned Dataset
from datasets import load_dataset

dataset = load_dataset("nicholas-ugbala-hf/chatdoctor-cleaned-10k")
train_dataset = dataset['train'].select(range(5000))
eval_dataset = dataset['eval']

print(f"Train: {len(train_dataset)}")
print(f"Eval: {len(eval_dataset)}")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/11.7M [00:00<?, ?B/s]

data/eval-00000-of-00001.parquet:   0%|          | 0.00/1.30M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9000 [00:00<?, ? examples/s]

Generating eval split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Train: 5000
Eval: 1000


In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "meta-llama/Llama-3.2-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading model...")

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.add_special_tokens({'pad_token': '<pad>'})
model.resize_token_embeddings(len(tokenizer))
tokenizer.padding_side = "right"
tokenizer.model_max_length = 512

print("Model loaded.")

Loading model...


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Model loaded.


In [7]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 9,175,040 || all params: 3,221,927,936 || trainable%: 0.2848


In [8]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Filter out any sequences over 512 tokens
def filter_long_sequences(sample):
    tokens = tokenizer(sample['text'], truncation=False)['input_ids']
    return len(tokens) <= 512

train_dataset = train_dataset.filter(filter_long_sequences)
eval_dataset  = eval_dataset.filter(filter_long_sequences)

print(f"Train after length filter: {len(train_dataset)}")
print(f"Eval after length filter:  {len(eval_dataset)}")

Filter:   0%|          | 0/5000 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (544 > 512). Running this sequence through the model will result in indexing errors


Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Train after length filter: 4937
Eval after length filter:  990


In [9]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="/kaggle/working/checkpoints",
    num_train_epochs=1,
    per_device_train_batch_size=4,       # back to 4
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,       # back to 4, effective batch = 16
    warmup_steps=50,
    learning_rate=2e-4,
    fp16=False,
    bf16=False,
    gradient_checkpointing=True,         # add this
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    load_best_model_at_end=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

print("Starting training...")
trainer.train()
print("Training complete.")

Adding EOS to train dataset:   0%|          | 0/4937 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/4937 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/990 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/990 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128256}.


Starting training...


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,2.570229,2.558034,2.557672,410828.000000,0.477830
200,2.524996,2.511456,2.530765,821552.000000,0.483773
300,2.482288,2.495902,2.482277,1232051.000000,0.485816


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(


Training complete.


In [10]:
from peft import PeftModel

model.eval()

def generate_response(prompt):
    messages = [
        {
            "role": "system",
            "content": "If you are a doctor, please answer the medical questions based on the patient's description."
        },
        {"role": "user", "content": prompt}
    ]

    encoded_inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
        padding=True
    ).to(model.device)

    outputs = model.generate(
        input_ids=encoded_inputs["input_ids"],
        attention_mask=encoded_inputs["attention_mask"],
        max_new_tokens=1000,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id
    )

    input_length = encoded_inputs["input_ids"].shape[-1]
    response = outputs[0][input_length:]
    return tokenizer.decode(response, skip_special_tokens=True)
    

In [11]:
questions = [
    "What are the early symptoms of type 2 diabetes?",
    "What is the first-line treatment for hypertension in adults?",
    "What causes iron deficiency anemia?",
    "How is malaria diagnosed and treated?",
    "What are the warning signs of a heart attack?"
]

print("FINE-TUNED MODEL OUTPUTS")
print("=" * 60)

for q in questions:
    print(f"Q: {q}")
    print(f"A: {generate_response(q)}")
    print("=" * 60)

FINE-TUNED MODEL OUTPUTS
Q: What are the early symptoms of type 2 diabetes?
A: Early symptoms of type 2 diabetes can include:
* Increased thirst and urination
* Fatigue or feeling weak
* Blurred vision
* Slow healing of cuts and wounds
* Tingling or numbness in hands and feet
* Unexplained weight loss
* Recurring infections
* Increased hunger
* Irritability or mood swings
* Headaches
* Unusual breath odor
It's essential to note that some people with type 2 diabetes may not exhibit any symptoms in the early stages. Therefore, it's crucial to get screened for diabetes at least once a year if you're over 45 years old, overweight, or have a family history of diabetes.
Consult with a healthcare provider if you're concerned about your health. They can help you determine if you have diabetes and develop a treatment plan to manage the condition.
Hope I answered your query. Let me know if I can assist you further.

Q: What is the first-line treatment for hypertension in adults?
A: According to 